# 립리딩 전처리 · 학습 파이프라인 (Colab)

원본 영상을 Google Drive에 두고, 코랩 GPU로 전처리와 학습을 수행한다.

**실행 전 준비**
1. 런타임 → 런타임 유형 변경 → 하드웨어 가속기 **GPU** 선택
2. Drive에 영상 폴더 생성 후 녹화본 업로드
3. 파일명 규칙: `{화자}_{문구}_{번호}.mp4` — 예) `s01_물주세요_01.mp4`

화자가 **2명 이상**이어야 학습이 진행된다. 화자 단위로 학습·검증을 나누기 때문이다.

## 1. 환경 확인

In [1]:
import torch

print(f"torch {torch.__version__}")
print(f"CUDA 사용 가능: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("경고: 런타임 유형을 GPU로 변경하세요.")

torch 2.11.0+cu128
CUDA 사용 가능: True
GPU: NVIDIA A100-SXM4-40GB


## 2. Drive 마운트

In [2]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


## 3. 경로 설정

`DRIVE_ROOT` 아래 `raw/`에 영상을 두면, 전처리 결과가 `processed/`에 저장된다.
Drive에 저장하므로 런타임이 끊겨도 전처리를 다시 하지 않아도 된다.

In [3]:
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/hanium-lipreading")
DRIVE_RAW = DRIVE_ROOT / "raw"
DRIVE_PROCESSED = DRIVE_ROOT / "processed"
DRIVE_CHECKPOINTS = DRIVE_ROOT / "checkpoints"

for folder in (DRIVE_RAW, DRIVE_PROCESSED, DRIVE_CHECKPOINTS):
    folder.mkdir(parents=True, exist_ok=True)

videos = sorted(
    p.name for p in DRIVE_RAW.glob("*") if p.suffix.lower() in (".mp4", ".avi", ".mov")
)
print(f"영상 {len(videos)}개")
for name in videos[:10]:
    print(f"  {name}")

영상 1234개
  s01_가래가있어요_01.mp4
  s01_가래가있어요_02.mp4
  s01_가래가있어요_03.mp4
  s01_가래가있어요_04.mp4
  s01_가래가있어요_05.mp4
  s01_가래가있어요_06.mp4
  s01_가래가있어요_07.mp4
  s01_가래가있어요_08.mp4
  s01_가래가있어요_09.mp4
  s01_가래가있어요_10.mp4


## 4. 저장소 clone

이미 받아둔 저장소가 있으면 `git pull`로 최신 코드만 가져온다.
clone이 중간에 실패해 빈 폴더가 남은 경우에는 지우고 다시 받는다.

비공개 저장소면 `https://<TOKEN>@github.com/...` 형태로 토큰을 넣는다.
토큰은 노트북에 저장하지 말고 매번 입력한다.

**코드를 갱신한 뒤에는 런타임을 다시 시작하거나 모듈을 reload해야 반영된다.**
파이썬은 한 번 import한 모듈을 다시 읽지 않는다.

```python
import importlib
import src.ml.training.dataset, src.ml.training.train
importlib.reload(src.ml.training.dataset)
importlib.reload(src.ml.training.train)
from src.ml.training.train import train
```

In [4]:
import os

REPO_URL = "https://github.com/HumanRhoid/hanium-lipreading.git"
BRANCH = "develop"
REPO_DIR = Path("/content/hanium-lipreading")

# clone이 중간에 실패하면 빈 폴더만 남아 다음 실행에서 git 명령이 어긋난다.
os.chdir("/content")
if (REPO_DIR / ".git").exists():
    !cd {REPO_DIR} && git fetch origin && git checkout {BRANCH} && git pull
else:
    !rm -rf {REPO_DIR}
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print(f"작업 경로: {Path.cwd()}")

Cloning into '/content/hanium-lipreading'...
remote: Enumerating objects: 658, done.
remote: Counting objects: 100% (215/215), done.
remote: Compressing objects: 100% (142/142), done.
remote: Total 658 (delta 91), reused 110 (delta 61), pack-reused 443 (from 1)
Receiving objects: 100% (658/658), 583.06 KiB | 5.30 MiB/s, done.
Resolving deltas: 100% (296/296), done.
작업 경로: /content/hanium-lipreading


## 5. 의존성 설치

`uv sync`는 쓰지 않는다. `pyproject.toml`이 torch를 **CPU 전용 인덱스**로 고정하고 있어
코랩의 GPU torch가 CPU 버전으로 교체되기 때문이다. 필요한 것만 pip로 설치한다.

In [5]:
!pip install --quiet mediapipe opencv-python wandb

import torch

print(f"설치 후 CUDA 사용 가능: {torch.cuda.is_available()}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.9/37.9 MB 74.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.4/137.4 kB 15.4 MB/s eta 0:00:00
설치 후 CUDA 사용 가능: True


## 6. 얼굴 랜드마크 모델 내려받기

`face_landmarker.task`는 `.gitignore`에 제외돼 있어 저장소에 없다.

In [6]:
LANDMARKER_URL = "https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task"
landmarker_path = REPO_DIR / "models" / "face_landmarker.task"
landmarker_path.parent.mkdir(parents=True, exist_ok=True)

if not landmarker_path.exists():
    !wget -q -O {landmarker_path} {LANDMARKER_URL}

print(f"{landmarker_path.name}: {landmarker_path.stat().st_size / 1e6:.1f} MB")

face_landmarker.task: 3.8 MB


## 7. 전처리 — 영상을 .npy로 변환

Drive를 입출력으로 직접 지정한다. 이미 변환된 파일은 건너뛴다.

In [7]:
import sys

sys.path.insert(0, str(REPO_DIR))

from src.ml.preprocess.vid2npy import run_batch

run_batch(raw_dir=DRIVE_RAW, processed_dir=DRIVE_PROCESSED)

이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed/s01_가래가있어요_01.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed/s01_가래가있어요_02.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed/s01_가래가있어요_03.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed/s01_가래가있어요_04.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed/s01_가래가있어요_05.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed/s01_가래가있어요_06.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed/s01_가래가있어요_07.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed/s01_가래가있어요_08.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed/s01_가래가있어요_09.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed/s01_가래가있어요_10.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed/s01_가래가있어요_11.npy
이미 존재함, 건너뜀: /content

In [8]:
FRAMES = 45
PROCESSED_F45 = DRIVE_ROOT / "processed_f45"

from src.ml.preprocess import vid2npy
from src.ml.preprocess.lip_crop import crop_lip_frames
from src.ml.preprocess.normalize import normalize_frames

# normalize_frames의 기본값은 def 시점에 고정되므로 함수를 갈아끼운다
def process_video_f45(video_path, landmarker):
    lips, opennesses = crop_lip_frames(video_path, landmarker)
    if not lips:
        return None
    return normalize_frames(lips, opennesses, fixed_frame_count=FRAMES)

vid2npy.process_video = process_video_f45
vid2npy.run_batch(raw_dir=DRIVE_RAW, processed_dir=PROCESSED_F45)

이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed_f45/s01_가래가있어요_01.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed_f45/s01_가래가있어요_02.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed_f45/s01_가래가있어요_03.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed_f45/s01_가래가있어요_04.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed_f45/s01_가래가있어요_05.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed_f45/s01_가래가있어요_06.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed_f45/s01_가래가있어요_07.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed_f45/s01_가래가있어요_08.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed_f45/s01_가래가있어요_09.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed_f45/s01_가래가있어요_10.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed_f45/s0

In [30]:
FRAMES = 60
PROCESSED_F60 = DRIVE_ROOT / "processed_f60"

from src.ml.preprocess import vid2npy
from src.ml.preprocess.lip_crop import crop_lip_frames
from src.ml.preprocess.normalize import normalize_frames

def process_video_f60(video_path, landmarker):
    lips, opennesses = crop_lip_frames(video_path, landmarker)
    if not lips:
        return None
    return normalize_frames(lips, opennesses, fixed_frame_count=FRAMES)

vid2npy.process_video = process_video_f60
vid2npy.run_batch(raw_dir=DRIVE_RAW, processed_dir=PROCESSED_F60)

처리 중: /content/drive/MyDrive/hanium-lipreading/raw/s01_가래가있어요_01.mp4
  저장 완료: /content/drive/MyDrive/hanium-lipreading/processed_f60/s01_가래가있어요_01.npy (shape=(60, 80, 112, 3))
처리 중: /content/drive/MyDrive/hanium-lipreading/raw/s01_가래가있어요_02.mp4
  저장 완료: /content/drive/MyDrive/hanium-lipreading/processed_f60/s01_가래가있어요_02.npy (shape=(60, 80, 112, 3))
처리 중: /content/drive/MyDrive/hanium-lipreading/raw/s01_가래가있어요_03.mp4
  저장 완료: /content/drive/MyDrive/hanium-lipreading/processed_f60/s01_가래가있어요_03.npy (shape=(60, 80, 112, 3))
처리 중: /content/drive/MyDrive/hanium-lipreading/raw/s01_가래가있어요_04.mp4
  저장 완료: /content/drive/MyDrive/hanium-lipreading/processed_f60/s01_가래가있어요_04.npy (shape=(60, 80, 112, 3))
처리 중: /content/drive/MyDrive/hanium-lipreading/raw/s01_가래가있어요_05.mp4
  저장 완료: /content/drive/MyDrive/hanium-lipreading/processed_f60/s01_가래가있어요_05.npy (shape=(60, 80, 112, 3))
처리 중: /content/drive/MyDrive/hanium-lipreading/raw

## 8. 매니페스트 생성

`.npy` 파일명을 파싱해 라벨과 화자를 뽑아낸다.

In [19]:
from scripts.build_manifest import build

manifest_path = DRIVE_ROOT / "manifest.csv"
build(processed_dir=DRIVE_PROCESSED, manifest_path=manifest_path)

매니페스트 생성: /content/drive/MyDrive/hanium-lipreading/manifest.csv
  클립 1234개 · 문구 15개 · 화자 8명
  라벨 매핑: 0=가래가있어요, 1=간호사불러주세요, 2=더워요, 3=도와주세요, 4=물주세요, 5=배고파요, 6=보호자불러주세요, 7=숨쉬기힘들어요, 8=아파요, 9=어지러워요, 10=자세바꿔주세요, 11=진통제주세요, 12=추워요, 13=토할거같아요, 14=화장실가고싶어요


In [20]:
import csv
from collections import Counter

with open(manifest_path, encoding="utf-8") as manifest_file:
    rows = list(csv.DictReader(manifest_file))

print(f"클립 {len(rows)}개")
print(f"화자별: {dict(Counter(row['speaker_id'] for row in rows))}")
print(f"문구별: {dict(Counter(row['label_text'] for row in rows))}")

클립 1234개
화자별: {'s01': 157, 's03': 148, 's04': 150, 's05': 150, 's06': 157, 's07': 171, 's08': 151, 's09': 150}
문구별: {'가래가있어요': 80, '간호사불러주세요': 79, '더워요': 86, '도와주세요': 82, '물주세요': 88, '배고파요': 85, '보호자불러주세요': 82, '숨쉬기힘들어요': 82, '아파요': 79, '어지러워요': 80, '자세바꿔주세요': 83, '진통제주세요': 86, '추워요': 82, '토할거같아요': 80, '화장실가고싶어요': 80}


### (45 프레임)


In [17]:
from scripts.build_manifest import build

manifest_f45 = DRIVE_ROOT / "manifest_f45.csv"
build(processed_dir=PROCESSED_F45, manifest_path=manifest_f45)

매니페스트 생성: /content/drive/MyDrive/hanium-lipreading/manifest_f45.csv
  클립 1234개 · 문구 15개 · 화자 8명
  라벨 매핑: 0=가래가있어요, 1=간호사불러주세요, 2=더워요, 3=도와주세요, 4=물주세요, 5=배고파요, 6=보호자불러주세요, 7=숨쉬기힘들어요, 8=아파요, 9=어지러워요, 10=자세바꿔주세요, 11=진통제주세요, 12=추워요, 13=토할거같아요, 14=화장실가고싶어요


### (60프레임 매니페스트 및 로컬 복사 및 검증포함)


In [38]:
from scripts.build_manifest import build
from pathlib import Path
import shutil, time, numpy as np

manifest_f60 = DRIVE_ROOT / "manifest_f60.csv"
build(processed_dir=PROCESSED_F60, manifest_path=manifest_f60)

LOCAL_F60 = Path("/content/data60")
target = LOCAL_F60 / "processed"
expected = len(list(PROCESSED_F60.glob("*.npy")))

if target.exists() and len(list(target.glob("*.npy"))) != expected:
    shutil.rmtree(target)
if not target.exists():
    t = time.time()
    shutil.copytree(PROCESSED_F60, target)
    print(f"복사 {time.time() - t:.0f}초")

# 아까 겪은 0바이트·누락 확인
bad = [p.name for p in target.glob("*.npy")
       if p.stat().st_size == 0 or (np.load(p, mmap_mode="r") is None)]
n = len(list(target.glob("*.npy")))
print(f"로컬 {n}/{expected}개 · 손상 {len(bad)}개")
assert n == expected and not bad, "복사가 불완전합니다"

TRAIN_ROOT_F60 = LOCAL_F60

매니페스트 생성: /content/drive/MyDrive/hanium-lipreading/manifest_f60.csv
  클립 1234개 · 문구 15개 · 화자 8명
  라벨 매핑: 0=가래가있어요, 1=간호사불러주세요, 2=더워요, 3=도와주세요, 4=물주세요, 5=배고파요, 6=보호자불러주세요, 7=숨쉬기힘들어요, 8=아파요, 9=어지러워요, 10=자세바꿔주세요, 11=진통제주세요, 12=추워요, 13=토할거같아요, 14=화장실가고싶어요
로컬 1234/1234개 · 손상 0개


## 8-1. 학습 데이터를 로컬 디스크로 복사

Drive 마운트는 네트워크 파일시스템이라 매 에폭 수백 개를 원격에서 읽는다.
런타임 로컬 디스크로 옮기면 GPU가 데이터를 기다리는 시간이 줄어든다.

런타임이 끊기면 사라지므로 세션마다 다시 실행한다. 복사에 1~2분 걸린다.

In [21]:
import shutil
import time

LOCAL_ROOT = Path("/content/data")
LOCAL_PROCESSED = LOCAL_ROOT / "processed"

started = time.time()
if not LOCAL_PROCESSED.exists():
    shutil.copytree(DRIVE_PROCESSED, LOCAL_PROCESSED)

# 매니페스트의 clip_path가 "processed/..." 라서 data_root만 바꾸면 그대로 맞는다.
TRAIN_ROOT = LOCAL_ROOT
local_count = len(list(LOCAL_PROCESSED.glob("*.npy")))
print(f"로컬 npy {local_count}개 · {time.time() - started:.0f}초")

로컬 npy 1230개 · 0초


### (45 프레임)


In [15]:
import shutil, time

LOCAL_F45 = Path("/content/data45")
target = LOCAL_F45 / "processed"      # 매니페스트가 "processed/..." 로 적으므로 이 이름이어야 함
expected = len(list(PROCESSED_F45.glob("*.npy")))

if target.exists() and len(list(target.glob("*.npy"))) != expected:
    shutil.rmtree(target)             # 개수 안 맞으면 다시 복사 (8-1 셀의 그 함정)
if not target.exists():
    started = time.time()
    shutil.copytree(PROCESSED_F45, target)
    print(f"복사 {time.time() - started:.0f}초")

TRAIN_ROOT_F45 = LOCAL_F45
print(f"로컬 npy {len(list(target.glob('*.npy')))}개 / 기대 {expected}개")

KeyboardInterrupt: 

## 9. 학습

체크포인트는 Drive에 저장되므로 런타임이 끊겨도 남는다.
학습 데이터는 `TRAIN_ROOT`(로컬 복사본)에서 읽어 I/O 대기를 줄인다.

**실험은 이 셀의 인자만 바꾸면 된다.** 저장소 코드를 고칠 필요가 없다.

- `seed` — 가중치 초기값·데이터 순서·증강을 한꺼번에 고정한다.
  같은 시드면 같은 결과가 나오므로 설정 비교의 전제가 된다
- `val_speakers` — 검증에 쓸 화자. **설정을 비교할 때는 반드시 고정한다.**
  `None`이면 화자 구성이 바뀔 때 검증 대상도 함께 바뀌어 비교가 깨진다
- `hidden_dim` / `num_layer` / `dropout` — 모델 크기와 정규화 강도
- `weight_decay` — 가중치를 작게 유지해 과적합을 억제
- `smoothing` — 최근 몇 에폭 평균으로 체크포인트를 판정할지. `1`이면 단일 에폭 최고치
- `label_smoothing` — 정답 확률을 100%로 몰지 않게 해 과신을 줄인다
- `grad_clip` — 드물게 튀는 그래디언트가 가중치를 흔드는 것을 막는다
- `ema_decay` — 가중치 이동평균으로 검증한다. 후반 진동이 완만해진다
- `augment` / `augmentation_config` — 증강 사용 여부와 강도
- `pretrained` — ImageNet 가중치로 백본을 초기화. 입력 정규화도 함께 바뀐다
- `freeze_backbone` — ResNet 층을 고정. `pretrained`와 함께 쓴다
- `amp` — bfloat16 혼합정밀도. GPU에서만 켜지고 속도가 2~3배 빨라진다
- `wandb_project` — 지정하면 실험이 웹 대시보드에 자동 기록된다

`label_smoothing` · `grad_clip` · `ema_decay`는 안정화 장치다. 셋 다 `0`을 주면
꺼진다. 체크포인트는 EMA를 켜면 평균 가중치로 저장되므로 검증 수치와 일치한다.

**시드 하나로 낸 결과는 그 자체로 성능이 아니다.** 같은 설정이라도 시드가 다르면
0.05~0.1 정도 흔들린다. 설정을 비교하거나 최종 수치를 낼 때는 시드 2~3개로
돌려 평균을 쓴다.

`pretrained=True`에 `freeze_backbone=False`면 학습률을 `3e-5`로 낮춘다. 좋은
초기값을 큰 보폭이 흐트러뜨리기 때문이다. 반대로 동결하면 움직이는 파라미터가
적어 `3e-4`까지 올려도 안정적이다.

**한 번에 하나만 바꾼다.** 두 개를 동시에 바꾸면 무엇이 효과였는지 알 수 없다.
`run_name`에 설정을 알아볼 수 있는 이름을 붙이면 나중에 비교하기 쉽다.

In [ ]:
from src.ml.preprocess.augmentation import AugmentationConfig
from src.ml.training.train import train

# 증강 강도를 조절하려면 설정을 만들어 넘긴다. None이면 기본값을 쓴다.
strong_augmentation = AugmentationConfig(
    brightness_probability=0.7,
    contrast_probability=0.7,
    rotation_probability=0.6,
    shift_probability=0.6,
    zoom_probability=0.6,
)

best_accuracy = train(
    manifest_path=manifest_path,
    data_root=TRAIN_ROOT,
    epochs=80,
    batch_size=16,
    learning_rate=2e-4,
    seed=42,  # 가중치 초기값·데이터 순서·증강을 함께 고정한다
    val_speakers=["s04"],  # 설정 비교 시 고정. None이면 seed로 무작위 선택
    checkpoint_path=DRIVE_CHECKPOINTS / "best.pt",
    num_workers= 8,
    amp=True,
    hidden_dim=300,
    num_layer=2,
    dropout=0.3,
    weight_decay=0.01,
    smoothing=3,
    label_smoothing=0.1,  # 0이면 끔. 정답에 대한 과신을 줄인다
    grad_clip=1.0,  # 0이면 끔. 튀는 그래디언트를 잘라낸다
    ema_decay=0.998,  # 0이면 끔. 가중치 이동평균으로 검증한다
    augment=True,
    augmentation_config=None,  # strong_augmentation 으로 바꿔 강도 실험
    pretrained=False,  # True면 ImageNet 가중치 + ImageNet 입력 정규화
    freeze_backbone=False,  # pretrained와 함께 켜면 ResNet 층을 고정한다
    wandb_project="lipreading",
    run_name="baseline_s04",
)

### (45 프레임)


In [ ]:
from src.ml.training.train import train

SEEDS = [42, 1, 7]
results45 = {}

for seed in SEEDS:
    print(f"\n{'='*16} s06 · seed {seed} · {FRAMES}프레임 {'='*16}")
    results45[seed] = train(
        manifest_path=manifest_f45,
        data_root=TRAIN_ROOT_F45,
        epochs=80,
        batch_size=16,
        learning_rate=2e-4,
        seed=seed,
        val_speakers=["s06"],
        checkpoint_path=DRIVE_CHECKPOINTS / f"f45_s06_seed{seed}.pt",
        num_workers=8,
        amp=True,
        ema_decay=0.998,
        hidden_dim=300,
        num_layer=2,
        dropout=0.3,
        smoothing=3,
        wandb_project="lipreading",
        run_name=f"f45_s06_seed{seed}",
    )

values = [results45[s] for s in SEEDS]
print(f"\n{'='*50}")
print(f"45프레임   {[f'{v:.3f}' for v in values]}")
print(f"           평균 {sum(values)/len(values):.3f} · 폭 {max(values)-min(values):.3f}")
print(f"30프레임   평균 0.754 · 폭 0.051")
print(f"판정선     0.804     ← 넘으면 채택")

### (60 프레임)

In [44]:
SEEDS = [42, 1, 7]
results60 = {}

for seed in SEEDS:
    print(f"\n{'='*16} s06 · seed {seed} · 60프레임 {'='*16}")
    results60[seed] = train(
        manifest_path=manifest_f60,
        data_root=TRAIN_ROOT_F60,
        epochs=80,
        batch_size=16,
        learning_rate=2e-4,
        seed=seed,
        val_speakers=["s06"],
        checkpoint_path=DRIVE_CHECKPOINTS / f"f60_s06_seed{seed}.pt",
        num_workers=8,
        amp=True,
        ema_decay=0.998,
        hidden_dim=300,
        num_layer=2,
        dropout=0.3,
        smoothing=3,
        wandb_project="lipreading",
        run_name=f"f60_s06_seed{seed}",
    )

v = [results60[s] for s in SEEDS]
print(f"\n{'='*54}")
print(f"60프레임   {[f'{x:.3f}' for x in v]}   평균 {sum(v)/3:.3f} · 폭 {max(v)-min(v):.3f}")
print(f"45프레임   ['0.707', '0.828', '0.694']   평균 0.743 · 폭 0.134")
print(f"30프레임   ['0.732', '0.783', '0.745']   평균 0.754 · 폭 0.051")
print(f"채택선     0.805")


================ s06 · seed 42 · 60프레임 ================


장치: cuda | 클래스: 15개
학습 1077개 · 검증 157개 클립 | 검증 화자 ['s06']
모델 hidden 300 · layer 2 · dropout 0.3 · wd 0.01 | 증강 켬 · 사전학습 끔 · 동결 끔
학습 파라미터 14.31M / 전체 14.31M | AMP 켬 · worker 8
label smoothing 0.1 · grad clip 1.0 · EMA 0.998 | 저장 기준 최근 3에폭 평균
[  1/80] train loss 2.7327 acc 0.085 | val loss 2.7087 acc 0.089 avg 0.089 | lr 2.00e-04
[  2/80] train loss 2.4717 acc 0.167 | val loss 2.7085 acc 0.064 avg 0.076 | lr 2.00e-04
[  3/80] train loss 2.2343 acc 0.275 | val loss 2.7097 acc 0.064 avg 0.072 | lr 1.99e-04
[  4/80] train loss 1.9036 acc 0.437 | val loss 2.7213 acc 0.083 avg 0.070 | lr 1.99e-04
[  5/80] train loss 1.5471 acc 0.639 | val loss 2.7606 acc 0.115 avg 0.087 | lr 1.98e-04
[  6/80] train loss 1.2729 acc 0.768 | val loss 2.8338 acc 0.070 avg 0.089 | lr 1.97e-04
[  7/80] train loss 1.0386 acc 0.858 | val loss 2.9488 acc 0.070 avg 0.085 | lr 1.96e-04
[  8/80] train loss 0.9151 acc 0.903 | val loss 3.0363 acc 0.070 avg 0.070 | lr 1.95e-04
[  9/80] train loss 0.8203 acc 0.934 | val loss

lr,█████▇▇▇▇▇▇▆▆▆▆▅▅▅▄▄▄▃▃▃▃▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁
train/acc,▁▃▅▆▇███████████████████████████████████
train/loss,█▇▆▅▄▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/acc,▁▁▁▁▁▂▂▃▃▄▆▆▇▇▇▇▇▇▇▇████████████████████
val/acc_smoothed,▁▁▁▁▁▁▂▂▂▃▄▄▅▆▆▇▇▇▇▇▇▇▇█████████████████
val/loss,▇▇▇▇▇█▇▇▆▆▄▄▃▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
best_val_acc,0.80892
best_val_acc_smoothed,0.81316
lr,0
train/acc,1
train/loss,0.56199



================ s06 · seed 1 · 60프레임 ================


장치: cuda | 클래스: 15개
학습 1077개 · 검증 157개 클립 | 검증 화자 ['s06']
모델 hidden 300 · layer 2 · dropout 0.3 · wd 0.01 | 증강 켬 · 사전학습 끔 · 동결 끔
학습 파라미터 14.31M / 전체 14.31M | AMP 켬 · worker 8
label smoothing 0.1 · grad clip 1.0 · EMA 0.998 | 저장 기준 최근 3에폭 평균
[  1/80] train loss 2.7344 acc 0.069 | val loss 2.7071 acc 0.070 avg 0.070 | lr 2.00e-04
[  2/80] train loss 2.4830 acc 0.165 | val loss 2.7059 acc 0.070 avg 0.070 | lr 2.00e-04
[  3/80] train loss 2.2625 acc 0.265 | val loss 2.7092 acc 0.070 avg 0.070 | lr 1.99e-04
[  4/80] train loss 2.0543 acc 0.382 | val loss 2.7221 acc 0.070 avg 0.070 | lr 1.99e-04
[  5/80] train loss 1.7209 acc 0.564 | val loss 2.7547 acc 0.070 avg 0.070 | lr 1.98e-04
[  6/80] train loss 1.4411 acc 0.683 | val loss 2.8124 acc 0.070 avg 0.070 | lr 1.97e-04
[  7/80] train loss 1.1709 acc 0.821 | val loss 2.9155 acc 0.070 avg 0.070 | lr 1.96e-04
[  8/80] train loss 1.0141 acc 0.859 | val loss 3.0333 acc 0.070 avg 0.070 | lr 1.95e-04
[  9/80] train loss 0.8986 acc 0.906 | val loss

lr,██████████▇▇▇▇▇▆▆▅▅▅▅▅▄▄▃▃▃▃▃▂▂▂▂▁▁▁▁▁▁▁
train/acc,▁▃▄▅▇███████████████████████████████████
train/loss,█▇▆▅▄▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/acc,▁▁▁▁▁▁▂▂▃▄▆▇▇▇▇█████████████████████████
val/acc_smoothed,▁▁▁▁▁▁▁▁▂▂▅▇▇▇▇█████████████████████████
val/loss,▆▆▇███▇▆▄▄▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
best_val_acc,0.80255
best_val_acc_smoothed,0.80255
lr,0
train/acc,1
train/loss,0.5629



================ s06 · seed 7 · 60프레임 ================


장치: cuda | 클래스: 15개
학습 1077개 · 검증 157개 클립 | 검증 화자 ['s06']
모델 hidden 300 · layer 2 · dropout 0.3 · wd 0.01 | 증강 켬 · 사전학습 끔 · 동결 끔
학습 파라미터 14.31M / 전체 14.31M | AMP 켬 · worker 8
label smoothing 0.1 · grad clip 1.0 · EMA 0.998 | 저장 기준 최근 3에폭 평균
[  1/80] train loss 2.7298 acc 0.084 | val loss 2.7067 acc 0.070 avg 0.070 | lr 2.00e-04
[  2/80] train loss 2.5383 acc 0.149 | val loss 2.7101 acc 0.070 avg 0.070 | lr 2.00e-04
[  3/80] train loss 2.2782 acc 0.264 | val loss 2.7185 acc 0.083 avg 0.074 | lr 1.99e-04
[  4/80] train loss 1.9746 acc 0.422 | val loss 2.7485 acc 0.083 avg 0.079 | lr 1.99e-04
[  5/80] train loss 1.6744 acc 0.588 | val loss 2.8094 acc 0.083 avg 0.083 | lr 1.98e-04
[  6/80] train loss 1.4138 acc 0.695 | val loss 2.9109 acc 0.083 avg 0.083 | lr 1.97e-04
[  7/80] train loss 1.1608 acc 0.820 | val loss 3.0686 acc 0.083 avg 0.083 | lr 1.96e-04
[  8/80] train loss 0.9524 acc 0.879 | val loss 3.2672 acc 0.083 avg 0.083 | lr 1.95e-04
[  9/80] train loss 0.8654 acc 0.922 | val loss

lr,████████▇▇▇▇▇▇▆▆▆▆▆▆▅▅▅▅▄▄▃▃▂▂▂▂▂▂▁▁▁▁▁▁
train/acc,▁▅▆▇▇███████████████████████████████████
train/loss,█▇▆▅▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/acc,▁▁▁▁▁▁▂▄▅▆▆▇▇▇▇▇▇▇▇▇████████████████████
val/acc_smoothed,▁▁▁▁▁▁▂▂▂▃▇▇▇▇▇▇████████████████████████
val/loss,▆▆▆▆▆▇███▇▄▄▃▃▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
best_val_acc,0.82166
best_val_acc_smoothed,0.82378
lr,0
train/acc,1
train/loss,0.56291



60프레임   ['0.809', '0.803', '0.822']   평균 0.811 · 폭 0.019
45프레임   ['0.707', '0.828', '0.694']   평균 0.743 · 폭 0.134
30프레임   ['0.732', '0.783', '0.745']   평균 0.754 · 폭 0.051
채택선     0.805


## 9-1. 교차검증

검증 화자 한 명으로 재면 그 사람의 난이도에 결과가 좌우된다. 화자를 바꿔가며
전부 한 번씩 검증으로 쓰고 평균을 내면 화자 편차에 흔들리지 않는 수치가 나온다.

**시드도 함께 반복해야 한다.** 같은 설정이라도 시드가 다르면 0.05~0.1 흔들리는데,
이 폭이 화자 간 차이와 비슷해서 한 번씩만 돌리면 순위를 신뢰할 수 없다.

`SEEDS`를 늘릴수록 신뢰도가 올라가지만 학습 횟수가 화자 수만큼 곱해진다.
경향만 볼 때는 시드 하나로, 발표에 쓸 최종 수치는 셋으로 돌린다.

In [ ]:
SEEDS = [42]  # [42, 1, 7] 로 늘리면 화자마다 여러 번 돌려 편차까지 본다

speakers = sorted({row["speaker_id"] for row in rows})
print(f"화자 {speakers} · 시드 {SEEDS}")

results = {}
for speaker in speakers:
    for seed in SEEDS:
        print(f"\n{'=' * 18} {speaker} · seed {seed} {'=' * 18}")
        results[(speaker, seed)] = train(
            manifest_path=manifest_path,
            data_root=TRAIN_ROOT,
            epochs=80,
            batch_size=16,
            learning_rate=2e-4,
            seed=seed,
            val_speakers=[speaker],
            checkpoint_path=DRIVE_CHECKPOINTS / f"cv_{speaker}_seed{seed}.pt",
            num_workers=8,
            amp=True,
            ema_decay=0.998,
            hidden_dim=300,
            num_layer=2,
            dropout=0.3,
            smoothing=3,
            wandb_project="lipreading",
            run_name=f"cv_{speaker}_seed{seed}",
        )

print(f"\n{'=' * 50}")
per_speaker = {}
for speaker in speakers:
    values = [results[(speaker, s)] for s in SEEDS]
    per_speaker[speaker] = sum(values) / len(values)
    detail = " ".join(f"{v:.3f}" for v in values)
    spread = f" (폭 {max(values) - min(values):.3f})" if len(values) > 1 else ""
    print(f"  {speaker}: {per_speaker[speaker]:.3f}   [{detail}]{spread}")

overall = sum(per_speaker.values()) / len(per_speaker)
gap = max(per_speaker.values()) - min(per_speaker.values())
print(f"\n전체 평균 {overall:.3f} · 화자 간 편차 {gap:.3f}")

## 10. 체크포인트 확인

In [ ]:
checkpoint = torch.load(DRIVE_CHECKPOINTS / "best.pt", map_location="cpu")

print(f"에폭 {checkpoint['epoch']}")
print(f"클래스 {checkpoint['num_classes']}개")
print(f"검증 정확도 {checkpoint['val_accuracy']:.3f}")
print(f"최근 평균 {checkpoint['smoothed_accuracy']:.3f}")
print(
    f"모델 hidden {checkpoint['hidden_dim']} · "
    f"layer {checkpoint['num_layer']} · dropout {checkpoint['dropout']}"
)

In [24]:
from pathlib import Path
import shutil, numpy as np

local = Path(TRAIN_ROOT) / "processed"

drive_names = {p.name for p in DRIVE_PROCESSED.glob("*.npy")}
local_names = {p.name for p in local.glob("*.npy")}
missing = drive_names - local_names
broken  = {p.name for p in local.glob("*.npy") if p.stat().st_size == 0}
fix = missing | broken

print(f"누락 {len(missing)} · 0바이트 {len(broken)} · 복구 대상 {len(fix)}개")
for name in sorted(fix):
    src = DRIVE_PROCESSED / name
    shutil.copy2(src, local / name)
    print(f"  {name}  ({src.stat().st_size:,}바이트)")

# 검증
bad = []
for p in local.glob("*.npy"):
    try:
        if p.stat().st_size == 0:
            raise ValueError
        np.load(p, mmap_mode="r")
    except Exception:
        bad.append(p.name)

print(f"\n로컬 {len(list(local.glob('*.npy')))}개 / Drive {len(drive_names)}개 · 손상 {len(bad)}개")
assert len(bad) == 0 and len(local_names | fix) == len(drive_names), "아직 안 맞습니다"

누락 4 · 0바이트 1 · 복구 대상 5개
  s03_숨쉬기힘들어요_03.npy  (806,528바이트)
  s03_숨쉬기힘들어요_04.npy  (806,528바이트)
  s03_숨쉬기힘들어요_05.npy  (806,528바이트)
  s03_숨쉬기힘들어요_06.npy  (806,528바이트)
  s03_숨쉬기힘들어요_07.npy  (806,528바이트)

로컬 1234개 / Drive 1234개 · 손상 0개


In [27]:
import torch, torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from src.ml.models.lip_reading_model import LipReadingModel
from src.ml.training.dataset import LipReadingDataset
from src.ml.training.train import split_by_speaker

SPEAKERS = ["s01", "s03", "s04", "s05", "s06", "s07", "s08", "s09"]
TAU_MAIN = 1.0          # 사전 선택값. 이걸로 판정한다

def probs_fold(speaker):
    ds = LipReadingDataset(manifest_path, TRAIN_ROOT)
    _, vi, _ = split_by_speaker(ds, val_speakers=[speaker])
    loader = DataLoader(Subset(ds, vi), batch_size=16, num_workers=2)
    ck = torch.load(DRIVE_CHECKPOINTS / f"cv_{speaker}_seed42.pt", map_location="cuda")
    m = LipReadingModel(num_classes=ck["num_classes"], hidden_dim=ck["hidden_dim"],
                        num_layer=ck["num_layer"], dropout=ck["dropout"]).cuda()
    m.load_state_dict(ck["model_state"]); m.eval()
    P, Y = [], []
    with torch.no_grad():
        for x, y in loader:
            with torch.autocast("cuda", dtype=torch.bfloat16):
                o = m(x.cuda())
            P.append(F.softmax(o.float(), 1).cpu()); Y.append(y)
    return torch.cat(P), torch.cat(Y), sorted({r["label_text"] for r in ds.rows})

fold = {}
for sp in SPEAKERS:
    fold[sp] = probs_fold(sp)
    print(f"{sp} 완료")

texts = fold["s01"][2]
C = len(texts)
total = sum(len(fold[s][1]) for s in SPEAKERS)

# ── 1. 예측 분포가 얼마나 치우쳤나 ──
pred_n, true_n = torch.zeros(C), torch.zeros(C)
for P, Y, _ in fold.values():
    pred_n += torch.bincount(P.argmax(1), minlength=C).float()
    true_n += torch.bincount(Y, minlength=C).float()

print(f"\n{'문구':<16}{'정답':>6}{'예측':>6}{'배율':>7}")
print("-" * 37)
for i in torch.argsort(pred_n / true_n, descending=True):
    print(f"{texts[i]:<16}{int(true_n[i]):>6}{int(pred_n[i]):>6}{pred_n[i] / true_n[i]:>7.2f}")

# ── 2. 보정 ──
def freq(speakers):
    n = torch.zeros(C)
    for s in speakers:
        n += torch.bincount(fold[s][0].argmax(1), minlength=C).float()
    return (n / n.sum()).clamp(min=1e-6)

base = sum((fold[s][0].argmax(1) == fold[s][1]).sum().item() for s in SPEAKERS) / total

print(f"\n{'τ':>5}{'상한(자기 화자)':>16}{'정직(타화자 추정)':>18}")
print("-" * 40)
for tau in (0.0, 0.25, 0.5, 0.75, 1.0):
    hi = ho = 0
    for sp in SPEAKERS:
        P, Y, _ = fold[sp]
        others = [s for s in SPEAKERS if s != sp]
        hi += ((P / freq([sp]) ** tau).argmax(1) == Y).sum().item()
        ho += ((P / freq(others) ** tau).argmax(1) == Y).sum().item()
    mark = "  ← 판정" if tau == TAU_MAIN else ""
    print(f"{tau:>5.2f}{hi / total:>16.3f}{ho / total:>18.3f}{mark}")

print(f"\n보정 없음  {base:.3f}")

# ── 3. τ=1 정직 버전, 화자별 ──
print(f"\n{'화자':<6}{'보정전':>8}{'보정후':>8}{'차이':>8}")
print("-" * 32)
for sp in SPEAKERS:
    P, Y, _ = fold[sp]
    others = [s for s in SPEAKERS if s != sp]
    b = (P.argmax(1) == Y).float().mean().item()
    a = ((P / freq(others) ** TAU_MAIN).argmax(1) == Y).float().mean().item()
    print(f"{sp:<6}{b:>8.3f}{a:>8.3f}{a - b:>+8.3f}")

s01 완료
s03 완료
s04 완료
s05 완료
s06 완료
s07 완료
s08 완료
s09 완료

문구                  정답    예측     배율
-------------------------------------
숨쉬기힘들어요             82   172   2.10
더워요                 86   151   1.76
가래가있어요              80   115   1.44
아파요                 79   100   1.27
토할거같아요              80    99   1.24
배고파요                85    99   1.16
어지러워요               80    71   0.89
간호사불러주세요            79    63   0.80
추워요                 82    64   0.78
물주세요                88    67   0.76
보호자불러주세요            82    62   0.76
진통제주세요              86    59   0.69
화장실가고싶어요            80    52   0.65
자세바꿔주세요             83    41   0.49
도와주세요               82    19   0.23

    τ       상한(자기 화자)        정직(타화자 추정)
----------------------------------------
 0.00           0.527             0.527
 0.25           0.529             0.523
 0.50           0.371             0.528
 0.75           0.378             0.525
 1.00           0.379             0.528  ← 판정

보정 없음  0.527

화자         보정전     보정후    

In [28]:
import collections

for target_name in ["도와주세요", "자세바꿔주세요", "숨쉬기힘들어요"]:
    t = texts.index(target_name)
    print(f"\n=== {target_name} ===")
    print(f"{'화자':<6}{'정답':>6}{'맞춤':>6}{'재현율':>8}  주요 오답")
    for sp in SPEAKERS:
        P, Y, _ = fold[sp]
        mask = Y == t
        n = int(mask.sum())
        if n == 0:
            continue
        pred = P[mask].argmax(1)
        hit = int((pred == t).sum())
        wrong = collections.Counter(texts[int(i)] for i in pred if int(i) != t)
        top = " · ".join(f"{a}×{c}" for a, c in wrong.most_common(2))
        print(f"{sp:<6}{n:>6}{hit:>6}{hit / n:>8.2f}  {top}")



=== 도와주세요 ===
화자        정답    맞춤     재현율  주요 오답
s01       11     3    0.27  물주세요×8
s03       10     1    0.10  보호자불러주세요×4 · 간호사불러주세요×2
s04       10     2    0.20  토할거같아요×7 · 어지러워요×1
s05       10     0    0.00  추워요×10
s06       11     6    0.55  토할거같아요×1 · 어지러워요×1
s07       10     1    0.10  토할거같아요×9
s08       10     2    0.20  어지러워요×6 · 숨쉬기힘들어요×2
s09       10     0    0.00  더워요×9 · 토할거같아요×1

=== 자세바꿔주세요 ===
화자        정답    맞춤     재현율  주요 오답
s01       10     7    0.70  물주세요×2 · 간호사불러주세요×1
s03       10     6    0.60  숨쉬기힘들어요×4
s04       10    10    1.00  
s05       10     0    0.00  배고파요×7 · 아파요×3
s06       11    11    1.00  
s07       12     0    0.00  가래가있어요×4 · 진통제주세요×4
s08       10     5    0.50  숨쉬기힘들어요×5
s09       10     0    0.00  더워요×7 · 아파요×3

=== 숨쉬기힘들어요 ===
화자        정답    맞춤     재현율  주요 오답
s01       11     2    0.18  물주세요×4 · 추워요×3
s03       10    10    1.00  
s04       10     6    0.60  물주세요×2 · 아파요×1
s05       10     0    0.00  더워요×4 · 추워요×4
s06       10    10    1.00  
s0

In [29]:
import numpy as np, csv, collections
from pathlib import Path

with open(manifest_path, encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

acc = collections.defaultdict(list)
for r in rows:
    a = np.load(Path(TRAIN_ROOT) / r["clip_path"])[:, :, :, 0].astype(np.float32)
    motion = np.abs(np.diff(a, axis=0)).mean()
    dup = np.mean([np.array_equal(a[i], a[i + 1]) for i in range(len(a) - 1)])
    acc[r["label_text"]].append((motion, dup))

print(f"{'문구':<16}{'움직임':>8}{'중복률':>8}{'클립':>6}")
print("-" * 40)
for m, ph, d, n in sorted((np.mean([x[0] for x in v]), ph,
                           np.mean([x[1] for x in v]), len(v))
                          for ph, v in acc.items()):
    print(f"{ph:<16}{m:>8.2f}{d:>8.2f}{n:>6}")

문구                   움직임     중복률    클립
----------------------------------------
더워요                 9.12    0.19    86
추워요                 9.31    0.13    82
가래가있어요             10.19    0.06    80
물주세요               10.32    0.12    88
도와주세요              10.76    0.13    82
어지러워요              11.00    0.06    80
아파요                11.30    0.09    79
숨쉬기힘들어요            12.07    0.05    82
토할거같아요             12.18    0.03    80
배고파요               12.62    0.09    85
진통제주세요             13.09    0.00    86
자세바꿔주세요            13.40    0.00    83
화장실가고싶어요           13.44    0.00    80
보호자불러주세요           13.53    0.00    82
간호사불러주세요           13.75    0.01    79


# 증강


In [42]:
import importlib, sys
from src.ml.preprocess.augmentation import pipeline

fresh = importlib.reload(pipeline)                    # 소스에서 원본을 새로 읽음
patched = sys.modules["src.ml.training.train"].VideoAugmentation
patched.__call__ = fresh.VideoAugmentation.__call__   # 학습 코드가 쥔 클래스에 되돌림
print("복구:", patched.__call__.__qualname__)          # VideoAugmentation.__call__ 이면 정상

복구: VideoAugmentation.__call__


In [43]:
# ═══ 시간축 증강 실험 · 이 셀 하나만 실행 (재실행 안전) ═══
from pathlib import Path
import numpy as np
from src.ml.preprocess.augmentation.pipeline import VideoAugmentation
from src.ml.training.train import train

DRIVE_ROOT        = globals().get("DRIVE_ROOT", Path("/content/drive/MyDrive/hanium-lipreading"))
DRIVE_CHECKPOINTS = globals().get("DRIVE_CHECKPOINTS", DRIVE_ROOT / "checkpoints")
manifest_path     = globals().get("manifest_path", DRIVE_ROOT / "manifest.csv")
TRAIN_ROOT        = globals().get("TRAIN_ROOT",
                    Path("/content/data") if Path("/content/data/processed").exists() else DRIVE_ROOT)
assert manifest_path.exists(), f"매니페스트 없음: {manifest_path}"

TIME_CROP_PROB = 0.5
TIME_CROP_MIN  = 0.75
SEEDS = [42, 1, 7]

# 원본을 클래스 속성에 한 번만 보관 → 몇 번 실행해도 진짜 원본이 유지된다
if not getattr(VideoAugmentation, "_taug_patched", False):
    VideoAugmentation._taug_orig = VideoAugmentation.__call__
    VideoAugmentation._taug_patched = True
ORIG = VideoAugmentation._taug_orig
assert ORIG.__qualname__ == "VideoAugmentation.__call__", f"원본이 아님: {ORIG.__qualname__}"

def call_with_time_aug(self, clip, return_details=False):
    if return_details:
        return ORIG(self, clip, True)
    frames = ORIG(self, clip, False)
    if self.rng.random() < TIME_CROP_PROB:
        T = len(frames)
        keep = int(T * self.rng.uniform(TIME_CROP_MIN, 1.0))
        if 2 <= keep < T:
            start = int(self.rng.integers(0, T - keep + 1))
            idx = np.linspace(start, start + keep - 1, T).round().astype(int)
            frames = frames[idx]
    return frames

VideoAugmentation.__call__ = call_with_time_aug
print(f"시간축 증강 적용 · 확률 {TIME_CROP_PROB} · 크롭 하한 {TIME_CROP_MIN}")
print(f"데이터 {TRAIN_ROOT}\n")

results = {}
try:
    for seed in SEEDS:
        print(f"\n{'='*16} s06 · seed {seed} · 시간축 증강 {'='*16}")
        results[seed] = train(
            manifest_path=manifest_path, data_root=TRAIN_ROOT,
            epochs=80, batch_size=16, learning_rate=2e-4,
            seed=seed, val_speakers=["s06"],
            checkpoint_path=DRIVE_CHECKPOINTS / f"taug_s06_seed{seed}.pt",
            num_workers=8, amp=True, ema_decay=0.998,
            hidden_dim=300, num_layer=2, dropout=0.3, smoothing=3,
            wandb_project="lipreading", run_name=f"taug_s06_seed{seed}",
        )
finally:
    VideoAugmentation.__call__ = ORIG
    print("\n[증강 원상복구 완료]")

v = [results[s] for s in SEEDS if s in results]
print(f"\n{'='*56}")
if len(v) == len(SEEDS):
    print(f"시간축 증강   {[f'{x:.3f}' for x in v]}   평균 {sum(v)/3:.3f} · 폭 {max(v)-min(v):.3f}")
else:
    print(f"완료 {len(v)}/{len(SEEDS)}시드   {[f'{x:.3f}' for x in v]}")
print(f"기준선(30)   ['0.732', '0.783', '0.745']   평균 0.754 · 폭 0.051")
print(f"채택선       0.805")

시간축 증강 적용 · 확률 0.5 · 크롭 하한 0.75
데이터 /content/data


================ s06 · seed 42 · 시간축 증강 ================


장치: cuda | 클래스: 15개
학습 1077개 · 검증 157개 클립 | 검증 화자 ['s06']
모델 hidden 300 · layer 2 · dropout 0.3 · wd 0.01 | 증강 켬 · 사전학습 끔 · 동결 끔
학습 파라미터 14.31M / 전체 14.31M | AMP 켬 · worker 8
label smoothing 0.1 · grad clip 1.0 · EMA 0.998 | 저장 기준 최근 3에폭 평균
[  1/80] train loss 2.7389 acc 0.079 | val loss 2.7088 acc 0.057 avg 0.057 | lr 2.00e-04
[  2/80] train loss 2.4779 acc 0.180 | val loss 2.7103 acc 0.057 avg 0.057 | lr 2.00e-04
[  3/80] train loss 2.1865 acc 0.306 | val loss 2.7198 acc 0.057 avg 0.057 | lr 1.99e-04
[  4/80] train loss 1.8731 acc 0.471 | val loss 2.7552 acc 0.057 avg 0.057 | lr 1.99e-04
[  5/80] train loss 1.5124 acc 0.640 | val loss 2.8232 acc 0.057 avg 0.057 | lr 1.98e-04
[  6/80] train loss 1.2885 acc 0.729 | val loss 2.9075 acc 0.083 avg 0.066 | lr 1.97e-04
[  7/80] train loss 1.0516 acc 0.838 | val loss 3.0139 acc 0.083 avg 0.074 | lr 1.96e-04
[  8/80] train loss 0.9567 acc 0.883 | val loss 3.0837 acc 0.083 avg 0.083 | lr 1.95e-04
[  9/80] train loss 0.8735 acc 0.906 | val loss

lr,████████▇▇▇▇▇▇▇▆▆▆▅▅▅▄▄▄▄▄▃▃▂▂▂▁▁▁▁▁▁▁▁▁
train/acc,▁▂▅▇▇███████████████████████████████████
train/loss,█▇▆▅▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/acc,▁▁▁▁▁▁▂▂▃▄▇▇▇▇██████████████████████████
val/acc_smoothed,▁▁▁▁▁▁▁▂▃▄▆▆▇▇▇▇████████████████████████
val/loss,▆▆▇▇▇███▆▅▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
best_val_acc,0.79618
best_val_acc_smoothed,0.79406
lr,0
train/acc,1
train/loss,0.56216



================ s06 · seed 1 · 시간축 증강 ================


장치: cuda | 클래스: 15개
학습 1077개 · 검증 157개 클립 | 검증 화자 ['s06']
모델 hidden 300 · layer 2 · dropout 0.3 · wd 0.01 | 증강 켬 · 사전학습 끔 · 동결 끔
학습 파라미터 14.31M / 전체 14.31M | AMP 켬 · worker 8
label smoothing 0.1 · grad clip 1.0 · EMA 0.998 | 저장 기준 최근 3에폭 평균
[  1/80] train loss 2.7493 acc 0.077 | val loss 2.7049 acc 0.070 avg 0.070 | lr 2.00e-04
[  2/80] train loss 2.5197 acc 0.154 | val loss 2.7048 acc 0.070 avg 0.070 | lr 2.00e-04
[  3/80] train loss 2.2548 acc 0.279 | val loss 2.7089 acc 0.070 avg 0.070 | lr 1.99e-04
[  4/80] train loss 1.9322 acc 0.433 | val loss 2.7195 acc 0.070 avg 0.070 | lr 1.99e-04
[  5/80] train loss 1.6113 acc 0.600 | val loss 2.7518 acc 0.083 avg 0.074 | lr 1.98e-04
[  6/80] train loss 1.3672 acc 0.713 | val loss 2.8038 acc 0.083 avg 0.079 | lr 1.97e-04
[  7/80] train loss 1.1367 acc 0.796 | val loss 2.8853 acc 0.083 avg 0.083 | lr 1.96e-04
[  8/80] train loss 0.9655 acc 0.880 | val loss 2.9845 acc 0.083 avg 0.083 | lr 1.95e-04
[  9/80] train loss 0.8785 acc 0.905 | val loss

lr,██████▇▇▇▇▇▇▇▆▆▆▆▆▅▅▅▄▄▄▄▃▃▃▂▂▂▂▂▂▁▁▁▁▁▁
train/acc,▁▂▃▄▅▆▇█████████████████████████████████
train/loss,█▇▆▅▄▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/acc,▁▁▁▁▁▁▁▁▂▃▅▅▆▇▇▇████████████████████████
val/acc_smoothed,▁▁▁▁▁▁▂▂▃▃▅▅▇▇▇▇▇▇██████████████████████
val/loss,▇▇▇▇█▇▆▆▅▄▃▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
best_val_acc,0.80892
best_val_acc_smoothed,0.80892
lr,0
train/acc,1
train/loss,0.56251



================ s06 · seed 7 · 시간축 증강 ================


장치: cuda | 클래스: 15개
학습 1077개 · 검증 157개 클립 | 검증 화자 ['s06']
모델 hidden 300 · layer 2 · dropout 0.3 · wd 0.01 | 증강 켬 · 사전학습 끔 · 동결 끔
학습 파라미터 14.31M / 전체 14.31M | AMP 켬 · worker 8
label smoothing 0.1 · grad clip 1.0 · EMA 0.998 | 저장 기준 최근 3에폭 평균
[  1/80] train loss 2.7281 acc 0.096 | val loss 2.7053 acc 0.083 avg 0.083 | lr 2.00e-04
[  2/80] train loss 2.5178 acc 0.170 | val loss 2.7134 acc 0.070 avg 0.076 | lr 2.00e-04
[  3/80] train loss 2.1859 acc 0.308 | val loss 2.7375 acc 0.070 avg 0.074 | lr 1.99e-04
[  4/80] train loss 1.8723 acc 0.451 | val loss 2.7837 acc 0.070 avg 0.070 | lr 1.99e-04
[  5/80] train loss 1.6017 acc 0.589 | val loss 2.8461 acc 0.083 avg 0.074 | lr 1.98e-04
[  6/80] train loss 1.3332 acc 0.734 | val loss 2.8947 acc 0.083 avg 0.079 | lr 1.97e-04
[  7/80] train loss 1.1418 acc 0.799 | val loss 2.9304 acc 0.102 avg 0.089 | lr 1.96e-04
[  8/80] train loss 0.9542 acc 0.877 | val loss 2.9218 acc 0.121 avg 0.102 | lr 1.95e-04
[  9/80] train loss 0.8978 acc 0.899 | val loss

lr,█████████▇▇▇▇▇▇▆▆▆▅▅▅▅▄▄▄▄▃▃▃▂▂▂▂▂▁▁▁▁▁▁
train/acc,▁▂▅▆▆▇██████████████████████████████████
train/loss,█▇▆▅▄▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/acc,▁▁▁▁▁▃▃▄▄▄▆▆▇▇▇▇████████████████████████
val/acc_smoothed,▁▁▁▂▄▅▆▇▇▇██████████████████████████████
val/loss,▇████▆▅▅▅▄▄▃▃▃▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▂▂▂▂▂▂▂▁▁
best_val_acc,0.71975
best_val_acc_smoothed,0.71975
lr,0
train/acc,1
train/loss,0.5621



[증강 원상복구 완료]

시간축 증강   ['0.796', '0.809', '0.720']   평균 0.775 · 폭 0.089
기준선(30)   ['0.732', '0.783', '0.745']   평균 0.754 · 폭 0.051
채택선       0.805


#오디오추출


In [45]:
import subprocess, numpy as np, time
from pathlib import Path
import torch, torchaudio

MEL_DIR = DRIVE_ROOT / "mel"
MEL_DIR.mkdir(exist_ok=True)
SR, N_MELS, T_FIX = 16000, 64, 250      # 250프레임 ≈ 2.5초

melspec = torchaudio.transforms.MelSpectrogram(
    sample_rate=SR, n_fft=400, hop_length=160, n_mels=N_MELS)

def load_audio(path):
    out = subprocess.run(
        ["ffmpeg", "-v", "error", "-i", str(path),
         "-f", "f32le", "-ac", "1", "-ar", str(SR), "-"],
        capture_output=True).stdout
    return np.frombuffer(out, dtype=np.float32).copy()

videos = sorted(DRIVE_RAW.glob("*.mp4"))
done = failed = skipped = 0
started = time.time()

for p in videos:
    out_path = MEL_DIR / f"{p.stem}.npy"
    if out_path.exists():
        skipped += 1
        continue
    wav = load_audio(p)
    if wav.size < SR // 10:                      # 0.1초 미만이면 실패로 본다
        print(f"  오디오 없음: {p.name}")
        failed += 1
        continue
    m = melspec(torch.from_numpy(wav))           # (64, T)
    m = torch.log(m + 1e-6)
    if m.shape[1] < T_FIX:                       # 길이 고정
        m = torch.nn.functional.pad(m, (0, T_FIX - m.shape[1]))
    else:
        s = (m.shape[1] - T_FIX) // 2
        m = m[:, s:s + T_FIX]
    np.save(out_path, m.numpy().astype(np.float16))
    done += 1

print(f"\n추출 {done} · 건너뜀 {skipped} · 실패 {failed} · {time.time()-started:.0f}초")
print(f"멜 파일 {len(list(MEL_DIR.glob('*.npy')))}개 / 영상 {len(videos)}개")


추출 1234 · 건너뜀 0 · 실패 0 · 196초
멜 파일 1234개 / 영상 1234개


In [47]:
import numpy as np, torch, torch.nn as nn, csv
from pathlib import Path
from torch.utils.data import Dataset, DataLoader

# ── 데이터 점검 ──
for f in sorted(MEL_DIR.glob("*.npy"))[:5]:
    m = np.load(f).astype(np.float32)
    print(f"{f.name[:24]:26} {m.shape}  min {m.min():7.2f}  max {m.max():7.2f}  std {m.std():.2f}")

with open(manifest_path, encoding="utf-8") as f:
    rows = list(csv.DictReader(f))
labels = sorted({r["label_text"] for r in rows})
lab2id = {t: i for i, t in enumerate(labels)}
HOLD_OUT = "s06"

class MelSet(Dataset):
    def __init__(self, rows): self.rows = rows
    def __len__(self): return len(self.rows)
    def __getitem__(self, i):
        r = self.rows[i]
        m = np.load(MEL_DIR / Path(r["clip_path"]).name).astype(np.float32)
        m = (m - m.mean()) / (m.std() + 1e-6)          # ← 정규화 추가
        return torch.from_numpy(m).unsqueeze(0), lab2id[r["label_text"]]

usable = [r for r in rows if (MEL_DIR / Path(r["clip_path"]).name).exists()]
tr = [r for r in usable if r["speaker_id"] != HOLD_OUT]
va = [r for r in usable if r["speaker_id"] == HOLD_OUT]

torch.manual_seed(42)
teacher = nn.Sequential(
    nn.Conv2d(1, 32, 3, padding=1),  nn.BatchNorm2d(32),  nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64),  nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(64, 128, 3, padding=1),nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
    nn.AdaptiveAvgPool2d((1, 8)),                          # ← 시간축 8칸 유지
    nn.Flatten(), nn.Dropout(0.3), nn.Linear(128 * 8, len(labels)),
).cuda()

opt = torch.optim.AdamW(teacher.parameters(), lr=1e-3, weight_decay=0.01)
crit = nn.CrossEntropyLoss(label_smoothing=0.1)
tl = DataLoader(MelSet(tr), batch_size=32, shuffle=True, num_workers=2)
vl = DataLoader(MelSet(va), batch_size=32, num_workers=2)

for ep in range(1, 41):
    teacher.train(); th = tn = 0
    for x, y in tl:
        out = teacher(x.cuda()); loss = crit(out, y.cuda())
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
        th += (out.argmax(1).cpu() == y).sum().item(); tn += len(y)
    teacher.eval(); vh = vn = 0
    with torch.no_grad():
        for x, y in vl:
            vh += (teacher(x.cuda()).argmax(1).cpu() == y).sum().item(); vn += len(y)
    if ep % 5 == 0 or ep == 1:
        print(f"[{ep:2}/40] 학습 {th/tn:.3f} · 검증 {vh/vn:.3f}")

s01_가래가있어요_01.npy   (64, 250)  min  -13.81  max    3.32  std 4.54
s01_가래가있어요_02.npy   (64, 250)  min  -13.81  max    5.68  std 4.64
s01_가래가있어요_03.npy   (64, 250)  min  -13.81  max    3.70  std 5.11
s01_가래가있어요_04.npy   (64, 250)  min  -13.81  max    4.31  std 5.16
s01_가래가있어요_05.npy   (64, 250)  min  -13.81  max    4.26  std 5.22
[ 1/40] 학습 0.070 · 검증 0.070
[ 5/40] 학습 0.358 · 검증 0.146
[10/40] 학습 0.558 · 검증 0.287
[15/40] 학습 0.679 · 검증 0.236
[20/40] 학습 0.770 · 검증 0.312
[25/40] 학습 0.821 · 검증 0.210
[30/40] 학습 0.833 · 검증 0.172
[35/40] 학습 0.883 · 검증 0.178
[40/40] 학습 0.923 · 검증 0.210


In [49]:
import difflib, random, csv, subprocess, numpy as np
from pathlib import Path

with open(manifest_path, encoding="utf-8") as f:
    rows = list(csv.DictReader(f))
LABELS = sorted({r["label_text"] for r in rows})

def load_audio(stem):
    out = subprocess.run(["ffmpeg", "-v", "error", "-i", str(DRIVE_RAW / f"{stem}.mp4"),
                          "-f", "f32le", "-ac", "1", "-ar", "16000", "-"],
                         capture_output=True).stdout
    return np.frombuffer(out, dtype=np.float32).copy()

def nearest(text):
    t = text.replace(" ", "").replace(".", "")
    return max(LABELS, key=lambda L: difflib.SequenceMatcher(None, t, L).ratio())

sample = random.Random(0).sample([r for r in rows if r["speaker_id"] == "s06"], 40)
exact = near = 0
for r in sample:
    txt = asr(load_audio(Path(r["clip_path"]).stem))["text"]
    t = txt.replace(" ", "").replace(".", "")
    n = nearest(txt)
    exact += (t == r["label_text"])
    near  += (n == r["label_text"])
    if n != r["label_text"]:
        print(f"X  정답 {r['label_text']:<12} 전사 {t:<16} 최근접 {n}")

print(f"\n완전일치 {exact}/40 ({exact/40:.2f})")
print(f"최근접   {near}/40 ({near/40:.2f})   ← 이게 실제 성능")

X  정답 추워요          전사 좋아요              최근접 아파요
X  정답 더워요          전사 다와요              최근접 도와주세요
X  정답 추워요          전사 좋아요              최근접 아파요

완전일치 29/40 (0.72)
최근접   37/40 (0.93)   ← 이게 실제 성능
